In [1]:
# import libary
from datasets import load_dataset

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# laod the dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations")

In [3]:
# overview of the dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})

In [4]:
# One sample
dataset["train"][0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

In [5]:
# rename colums with meaningful keys
dataset = dataset["train"].rename_columns({"Context":"query","Response":"response"})

In [6]:
# One sample
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow

In [7]:
# remove empty queries and responses
dataset = dataset.filter(lambda sample: sample["query"] is not None and sample["query"].strip() != "")
dataset = dataset.filter(lambda sample: sample["response"] is not None and sample["response"].strip() != "")


In [8]:
# compute length
def compute_length(sample, new_key, field_key):
    return {new_key: len(sample[field_key].split(" "))}

In [9]:
# Adding length property for both queries and respnses
dataset = dataset.map(lambda sample: compute_length(sample, new_key="query_length", field_key="query"))
dataset = dataset.map(lambda sample: compute_length(sample, new_key="response_length", field_key="response"))

In [10]:
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow

In [11]:
# removing short response
dataset = dataset.filter(lambda x:x["response_length"] > 30)
len(dataset)

3437

In [12]:
sys_prompt = "You are a compassionate mental health support assistant who responds in a warm, empathetic, and non-judgmental way, validating the user’s feelings, offering practical and evidence-based coping suggestions, encouraging professional help when appropriate, providing crisis resources only if there is self-harm or suicide risk, and tailoring hopeful guidance to the user’s specific situation."

In [13]:
def add_sys_prompt(example):
    return {"instruction": sys_prompt}

In [14]:
dataset = dataset.map(add_sys_prompt)

In [15]:
# clean new line and extra spacing from the query and response
import re

def clean_text(text):
    """Clean extra whitespace, newlines, and normalize spacing"""
    if not text:
        return text
    
    # Replace multiple whitespace (including newlines, tabs) with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

def clean_dataset_text(example):
    """Clean query and response text"""
    return {
        "query": clean_text(example["query"]),
        "response": clean_text(example["response"])
    }

In [16]:
dataset = dataset.map(clean_dataset_text)

Map: 100%|██████████| 3437/3437 [00:00<00:00, 6262.46 examples/s]


In [17]:
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Ba

In [18]:
from transformers import AutoTokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [19]:
def prepare_dataset_with_text_column(dataset):
   
    
    # Data prompt
    data_prompt = """{}

### Input:
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        sys_prompts = examples["instruction"]
        queries = examples["query"]
        responses = examples["response"]
        texts = []
        
        for sys_prompt, query, response in zip(sys_prompts, queries, responses):
            text = data_prompt.format(sys_prompt, query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [20]:
dataset = prepare_dataset_with_text_column(dataset)

Map: 100%|██████████| 3437/3437 [00:00<00:00, 33620.32 examples/s]


In [21]:
print(dataset[50]["text"])

You are a compassionate mental health support assistant who responds in a warm, empathetic, and non-judgmental way, validating the user’s feelings, offering practical and evidence-based coping suggestions, encouraging professional help when appropriate, providing crisis resources only if there is self-harm or suicide risk, and tailoring hopeful guidance to the user’s specific situation.

### Input:
Pateint Query: I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years. I’ve never had counseling about any of this. Do I have too many issues to address in counseling?

### Response:
Usually people who struggle with depression also have symptoms of anxiety. These symptoms come about as a result of underlying causes. The obstacles you have been overcoming such as abuse, cancer an

In [24]:
dataset_full = dataset.shuffle(seed=42).train_test_split(train_size=0.7)

In [26]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 2405
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 1032
    })
})

In [27]:
dataset_train_validation = dataset_full["test"]
dataset_train_validation = dataset_train_validation.train_test_split(test_size=0.5, seed=42)

In [28]:
dataset_train_validation

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 516
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 516
    })
})

In [29]:
dataset_full["test"] = dataset_train_validation["train"]
dataset_full["validation"] = dataset_train_validation["test"]

In [30]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 2405
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 516
    })
    validation: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'instruction', 'text'],
        num_rows: 516
    })
})

In [31]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset_full.save_to_disk(f"{dataset_based_path}/mh-amod-conversation-v1")

Saving the dataset (1/1 shards): 100%|██████████| 516/516 [00:00<00:00, 33736.98 examples/s]


In [33]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh-amod-conversation-v1"
dataset = load_from_disk(dataset_based_path)

In [37]:
dataset["test"][100]

{'query': "Our relationship ended about 7 years ago, but don't know how to let go. How can I get over that person and move on?",
 'response': "What you're most likely trying to get over is to be attracted to the qualities which you felt attracted to in the person.The problem isn't releasing yourself from the person, per se. Letting go of a person means separating yourself from the qualities in a partner which you value.Two possibilities exist.Either you continue to feel strongly that the qualities in the past relationship are still meaningful and you'd like to base your next relationship on these qualities.Or, the person had qualities which are no longer relevant or necessary in your life and this is the reason the relationship ended.It is possible that even if you had an adequate exposure to certain qualities in someone that you simply miss the companionship of a relationship.See if you can figure out whether you need the qualities this person gave you or whether you simply like being

In [36]:
dataset["train"][6]

{'query': 'My ex-boyfriend boyfriend and I lived together. He had a two year affair with a girl and had three pregnancies with her. One was an abortion, another was a miscarriage, and then she had the third baby. They are not together, but he continues to contact me and wants me back. He has nothing to offer me.',
 'response': 'Hi Attica,This is a question I think a lot of people deal with...they feel confusion about why they can\'t forget about or get over (or stop connecting with) someone who they absolutely know isn\'t good for them. In your case, part of the problem is that he keeps trying to get back in touch with you. For some people, honestly, it\'s a game... to see how much power they have over you or it\'s their need to control you. If you don\'t want contact from this person, it\'s really important to give him clear messages about boundaries ("Don\'t contact me again"), and then ignore all of their communications. Any interactions or responses from you at all will feed their 